# 02 - Nettoyage des données Olist

**Entrée** : `data/raw/olist/*.csv`  
**Sortie** : tables propres dans `data/processed/` (CSV).

Chaque règle de nettoyage est justifiée et l'impact (lignes avant/après) est affiché, pour pouvoir le citer dans le rapport.

## 1. Configuration et chargement

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Remonte les dossiers jusqu'à trouver requirements.txt (racine du projet)."""
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists():
            return p
    raise FileNotFoundError("Racine du projet introuvable (requirements.txt)")


ROOT = find_project_root()
RAW = ROOT / "data" / "raw" / "olist"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Racine :", ROOT)

FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

ZIP_COLS = ["customer_zip_code_prefix", "seller_zip_code_prefix", "geolocation_zip_code_prefix"]
raw = {}
for name, fname in FILES.items():
    header = pd.read_csv(RAW / fname, nrows=0).columns
    dtype = {c: "string" for c in header if c in ZIP_COLS}
    raw[name] = pd.read_csv(RAW / fname, dtype=dtype)

clean = {}  # tables nettoyées
report = []  # journal des modifications


def log(table, rule, before, after):
    report.append({"table": table, "règle": rule, "avant": before, "après": after, "supprimées": before - after})

Racine : c:\Users\DELL\Downloads\projet_ai


## 2. Customers

- Suppression des doublons sur `customer_id`.
- Normalisation du texte (ville en minuscules sans espaces superflus, état en majuscules).

In [2]:
df = raw["customers"].copy()
n0 = len(df)
df = df.drop_duplicates(subset="customer_id")
df["customer_city"] = df["customer_city"].str.strip().str.lower()
df["customer_state"] = df["customer_state"].str.strip().str.upper()
df["customer_zip_code_prefix"] = df["customer_zip_code_prefix"].str.zfill(5)
log("customers", "doublons customer_id", n0, len(df))
clean["customers"] = df
df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## 3. Orders

- Conversion des colonnes de dates en `datetime`.
- Doublons sur `order_id`.
- Ajout de `is_delivered` et de `delivery_days` (livraison réelle).
- Les commandes `canceled` et `unavailable` sont **conservées** dans cette table (elles seront filtrées au moment du calcul des ventes).

In [3]:
df = raw["orders"].copy()
n0 = len(df)
df = df.drop_duplicates(subset="order_id")
log("orders", "doublons order_id", n0, len(df))

DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], errors="coerce")

df["order_status"] = df["order_status"].str.strip().str.lower()
df["is_delivered"] = df["order_status"].eq("delivered")
df["delivery_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Incohérence : livraison avant l'achat
bad = df["delivery_days"] < 0
print("Livraisons avant la date d'achat :", int(bad.sum()))
df.loc[bad, "delivery_days"] = pd.NA

clean["orders"] = df
df[["order_id", "order_status", "order_purchase_timestamp", "delivery_days"]].head()

Livraisons avant la date d'achat : 0


,order_id,order_status,order_purchase_timestamp,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33,8.436574
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,13.782037
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49,9.394213
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06,13.208750
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39,2.873877


## 4. Products

- Correction des noms de colonnes mal orthographiés d'Olist (`lenght` -> `length`).
- Ajout de la catégorie en anglais via `category_translation`.
- Catégorie manquante -> `unknown`.
- Dimensions/poids à 0 ou négatifs -> `NaN` (valeurs impossibles).

In [4]:
df = raw["products"].copy()
n0 = len(df)
df = df.drop_duplicates(subset="product_id")
log("products", "doublons product_id", n0, len(df))

df = df.rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length",
    }
)

trans = raw["category_translation"].drop_duplicates(subset="product_category_name")
df = df.merge(trans, on="product_category_name", how="left")

# Catégories sans traduction : on garde le nom portugais, sinon "unknown"
df["product_category"] = (
    df["product_category_name_english"]
    .fillna(df["product_category_name"])
    .fillna("unknown")
)
df = df.drop(columns=["product_category_name_english"])

for c in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    invalid = df[c] <= 0
    if invalid.any():
        print(f"{c} : {int(invalid.sum())} valeurs <= 0 mises à NaN")
    df.loc[invalid, c] = pd.NA

clean["products"] = df
print("Catégories distinctes :", df["product_category"].nunique())
df.head()

product_weight_g : 4 valeurs <= 0 mises à NaN
Catégories distinctes : 74


,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


## 5. Order items

- Doublons sur (`order_id`, `order_item_id`).
- `price` doit être > 0, `freight_value` >= 0.
- Conversion de `shipping_limit_date`.
- Suppression des lignes dont la commande, le produit ou le vendeur n'existe pas.

In [5]:
df = raw["items"].copy()
n0 = len(df)
df = df.drop_duplicates(subset=["order_id", "order_item_id"])
log("items", "doublons (order_id, order_item_id)", n0, len(df))

n0 = len(df)
df = df[(df["price"] > 0) & (df["freight_value"] >= 0)]
log("items", "price > 0 et freight_value >= 0", n0, len(df))

df["shipping_limit_date"] = pd.to_datetime(df["shipping_limit_date"], errors="coerce")

n0 = len(df)
df = df[
    df["order_id"].isin(clean["orders"]["order_id"])
    & df["product_id"].isin(clean["products"]["product_id"])
    & df["seller_id"].isin(raw["sellers"]["seller_id"])
]
log("items", "références orphelines (order/product/seller)", n0, len(df))

clean["items"] = df.reset_index(drop=True)
df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


## 6. Payments

- Doublons sur (`order_id`, `payment_sequential`).
- `payment_value` >= 0 ; les paiements `not_defined` sont signalés.

In [6]:
df = raw["payments"].copy()
n0 = len(df)
df = df.drop_duplicates(subset=["order_id", "payment_sequential"])
log("payments", "doublons (order_id, payment_sequential)", n0, len(df))

n0 = len(df)
df = df[df["payment_value"] >= 0]
log("payments", "payment_value >= 0", n0, len(df))

print(df["payment_type"].value_counts().to_string())
clean["payments"] = df.reset_index(drop=True)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3


## 7. Reviews

- Doublons sur `review_id` : Olist contient des `review_id` répétés pour des commandes différentes, on garde donc le couple (`review_id`, `order_id`).
- Conversion des dates.
- Un seul avis par commande : le plus récent.
- Les commentaires texte (très incomplets) sont conservés tels quels.

In [7]:
df = raw["reviews"].copy()
n0 = len(df)
df = df.drop_duplicates(subset=["review_id", "order_id"])
log("reviews", "doublons (review_id, order_id)", n0, len(df))

for c in ["review_creation_date", "review_answer_timestamp"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

n0 = len(df)
df = df.sort_values("review_answer_timestamp").drop_duplicates(subset="order_id", keep="last")
log("reviews", "un seul avis par commande (le plus récent)", n0, len(df))

n0 = len(df)
df = df[df["review_score"].between(1, 5)]
log("reviews", "review_score entre 1 et 5", n0, len(df))

clean["reviews"] = df.reset_index(drop=True)

## 8. Sellers

- Doublons sur `seller_id` et normalisation du texte.

In [8]:
df = raw["sellers"].copy()
n0 = len(df)
df = df.drop_duplicates(subset="seller_id")
df["seller_city"] = df["seller_city"].str.strip().str.lower()
df["seller_state"] = df["seller_state"].str.strip().str.upper()
df["seller_zip_code_prefix"] = df["seller_zip_code_prefix"].str.zfill(5)
log("sellers", "doublons seller_id", n0, len(df))
clean["sellers"] = df

## 9. Geolocation

Le fichier contient plusieurs points par code postal. On agrège à **une ligne par code postal** (médiane de la latitude/longitude), ce qui suffit pour les analyses par région.

In [9]:
df = raw["geolocation"].copy()
n0 = len(df)
df["geolocation_zip_code_prefix"] = df["geolocation_zip_code_prefix"].str.zfill(5)
df = (
    df.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        lat=("geolocation_lat", "median"),
        lng=("geolocation_lng", "median"),
        city=("geolocation_city", lambda s: s.str.strip().str.lower().mode().iat[0]),
        state=("geolocation_state", lambda s: s.str.strip().str.upper().mode().iat[0]),
    )
)
log("geolocation", "agrégation par code postal", n0, len(df))
clean["geolocation"] = df
df.head()

,geolocation_zip_code_prefix,lat,lng,city,state
0,01001,-23.550381,-46.634027,sao paulo,SP
1,01002,-23.548551,-46.635072,sao paulo,SP
2,01003,-23.548977,-46.635313,sao paulo,SP
3,01004,-23.549535,-46.634771,sao paulo,SP
4,01005,-23.549612,-46.636532,sao paulo,SP


## 10. Table de faits : lignes de vente

Une ligne par article vendu, avec date, produit, catégorie, vendeur, prix et frais de livraison. C'est la base du dashboard, du calcul de marge et de la prévision de demande.

On ne garde que les commandes **non annulées / non indisponibles**.

In [10]:
EXCLUDED_STATUSES = ["canceled", "unavailable"]

orders_ok = clean["orders"][~clean["orders"]["order_status"].isin(EXCLUDED_STATUSES)]
sales = (
    clean["items"]
    .merge(
        orders_ok[["order_id", "customer_id", "order_status", "order_purchase_timestamp"]],
        on="order_id",
        how="inner",
    )
    .merge(clean["products"][["product_id", "product_category"]], on="product_id", how="left")
    .merge(clean["customers"][["customer_id", "customer_state"]], on="customer_id", how="left")
)
sales["order_date"] = sales["order_purchase_timestamp"].dt.normalize()
sales["revenue"] = sales["price"]
sales["total_with_freight"] = sales["price"] + sales["freight_value"]

log("sales", "exclusion des commandes annulées / indisponibles", len(clean["items"]), len(sales))
clean["sales"] = sales
sales.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,product_category,customer_state,order_date,revenue,total_with_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,cool_stuff,RJ,2017-09-13,58.90,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,pet_shop,SP,2017-04-26,239.90,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,furniture_decor,MG,2018-01-14,199.00,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,perfumery,SP,2018-08-08,12.99,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,garden_tools,SP,2017-02-04,199.90,218.04


## 11. Contrôles finaux

In [11]:
assert clean["customers"]["customer_id"].is_unique
assert clean["orders"]["order_id"].is_unique
assert clean["products"]["product_id"].is_unique
assert clean["sellers"]["seller_id"].is_unique
assert not clean["items"].duplicated(["order_id", "order_item_id"]).any()
assert (clean["items"]["price"] > 0).all()
assert clean["sales"]["order_date"].notna().all()
print("Tous les contrôles sont OK.")

print("Période des ventes :", clean["sales"]["order_date"].min().date(), "->", clean["sales"]["order_date"].max().date())
print("Nombre de lignes de vente :", len(clean["sales"]))

Tous les contrôles sont OK.
Période des ventes : 2016-09-04 -> 2018-09-03
Nombre de lignes de vente : 112101


## 12. Journal des modifications

In [12]:
pd.DataFrame(report)

,table,règle,avant,après,supprimées
0,customers,doublons customer_id,99441,99441,0
1,orders,doublons order_id,99441,99441,0
2,products,doublons product_id,32951,32951,0
3,items,"doublons (order_id, order_item_id)",112650,112650,0
4,items,price > 0 et freight_value >= 0,112650,112650,0
5,items,références orphelines (order/product/seller),112650,112650,0
6,payments,"doublons (order_id, payment_sequential)",103886,103886,0
7,payments,payment_value >= 0,103886,103886,0
8,reviews,"doublons (review_id, order_id)",99224,99224,0
9,reviews,un seul avis par commande (le plus récent),99224,98673,551


## 13. Sauvegarde dans `data/processed/`

In [13]:
for name, df in clean.items():
    out = PROCESSED / f"{name}.csv"
    df.to_csv(out, index=False)
    print(f"{out.relative_to(ROOT)}  ({len(df):,} lignes)")

data\processed\customers.csv  (99,441 lignes)
data\processed\orders.csv  (99,441 lignes)
data\processed\products.csv  (32,951 lignes)
data\processed\items.csv  (112,650 lignes)
data\processed\payments.csv  (103,886 lignes)
data\processed\reviews.csv  (98,673 lignes)
data\processed\sellers.csv  (3,095 lignes)
data\processed\geolocation.csv  (19,015 lignes)
data\processed\sales.csv  (112,101 lignes)
